# How to make your own Agent

In this tutorial you'll learn:
- How to implement agents that work with TAG's hierarchy system
- The difference between BaseAgent and LevelAgent interfaces
- How to create standalone vs hierarchical agents
- Best practices for agent implementation

Each agent being part of a TAG hierarchy has to respect the `LevelAgent` interface from `src/tame/hierarchy/base_agent.py`.
An example of this is our `PPO` implementation from `src/tame/agents/monolithic_ppo.py`

In this tutorial we will discuss the important functions to implement for a well functioning agent.

As you can see, `src/tame/hierarchy/base_agent.py` provides two type of agent interfaces:
- `BaseAgent`: a general interface for any agent that the `run_experiment` script must run
- `LevelAgent`: the itnerface for agents to be part of a `LevelEnv`.

## BaseAgent

The `BaseAgent` class is a general interface for standalone RL agents that can be run using the `run_experiment.py` script. It requires implementing these core methods:

1. `__init__(env: ParallelEnv, args: None | Any = None)`:
   - Initializes the agent with a PettingZoo parallel environment
   - Takes optional arguments for customization
   - Should set up networks, memory buffers, and other agent components

2. `save_agent(save_path: str | Path, name: None | str = None)`:
   - Saves agent's state (networks, parameters) to disk
   - Models are saved in `{save_path}/models/{name}.pth`
   - Name parameter should not include '.pth' extension

3. `load_agent(load_path: Path | str, name: str = "trained_model") -> bool`:
   - Loads agent's state from disk
   - Returns True if loading was successful
   - Loads from `{load_path}/models/{name}.pth`

4. `act(observation: Dict[str, np.ndarray]) -> dict`:
   - Takes observations and returns actions for each agent
   - Input is dictionary: `{agent_name: observation}`
   - Output is dictionary: `{agent_name: action}`

5. `train(env: Any, log_path: Path | str | None = None, run_name: str | None = None)`:
   - Implements the training loop for the agent
   - Uses provided environment for training
   - Logs results to specified path if provided

Here's a minimal example:

```python
from tame.hierarchy import BaseAgent
from pathlib import Path
import torch
import numpy as np

class MyBaseAgent(BaseAgent):
    def __init__(self, env, args=None):
        self.env = env
        self.args = args or {}
        
        # Initialize networks, buffers etc.
        self.policy = torch.nn.Linear(
            np.prod(env.observation_spaces[env.possible_agents[0]].shape),
            env.action_spaces[env.possible_agents[0]].n
        )
        self.optimizer = torch.optim.Adam(self.policy.parameters())
        
    def save_agent(self, save_path, name=None):
        name = name or "model"
        save_dir = Path(save_path) / "models"
        save_dir.mkdir(parents=True, exist_ok=True)
        
        torch.save({
            'policy_state_dict': self.policy.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, save_dir / f"{name}.pth")
        
    def load_agent(self, load_path, name="trained_model"):
        try:
            checkpoint = torch.load(Path(load_path) / "models" / f"{name}.pth")
            self.policy.load_state_dict(checkpoint['policy_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            return True
        except:
            return False
            
    def act(self, observation):
        actions = {}
        for agent, obs in observation.items():
            with torch.no_grad():
                logits = self.policy(torch.FloatTensor(obs))
                action = torch.argmax(logits).item()
            actions[agent] = action
        return actions
        
    def train(self, env, log_path=None, run_name=None):
        # Your training loop here
        for episode in range(1000):
            obs, _ = env.reset()
            done = False
            while not done:
                actions = self.act(obs)
                next_obs, rewards, terminated, truncated, _ = env.step(actions)
                # Your update logic here
                done = any(terminated.values()) or any(truncated.values())
                obs = next_obs

### Note
`BaseAgent` is primarily useful if you want to use the `run_experiment.py` script. 
If you're building an agent to be part of a hierarchy, you should implement `LevelAgent` instead.

## LevelAgent

The `LevelAgent` interface inherits from `BaseAgent` but adds several key methods needed for hierarchical learning. When implementing your own agent, you'll need to:

1. Define the following class attributes:
   - `action_space`: Dict of action spaces for each agent the LevelAgent controls
   - `observation_space`: Dict of observation spaces from which the agent receives input
   - `communication_space`: Dict of spaces defining messages sent to higher levels (optional)

2. Implement these required methods:
   - `__init__(observation_space, action_space, communication_space, device, name, args, torch_compile)`:
     - Initializes spaces, networks, and memory buffers
     - Note: Unlike BaseAgent, LevelAgent doesn't take an environment in its constructor
   
   - `update_step(global_step, writer)`:
     - Performs a training update during environment steps
     - Called automatically by LevelEnv during training
     - Use writer (TensorBoard) to log training metrics
   
   - `store(state, action, reward, done)`:
     - Stores transitions in agent memory (e.g., replay buffer)
     - Called automatically by LevelEnv after each action
   
   - `act_train(observation, global_step)`:
     - Like `act()` but used during training
     - Typically implements exploration (e.g., epsilon-greedy)

   -  `save_agent(save_path: str | Path, name: None | str = None)`:
      - Saves agent's state (networks, parameters) to disk
      - Models are saved in `{save_path}/models/{name}.pth`
      - Name parameter should not include '.pth' extension

   -  `load_agent(load_path: Path | str, name: str = "trained_model") -> bool`:
      - Loads agent's state from disk
      - Returns True if loading was successful
      - Loads from `{load_path}/models/{name}.pth`

   - `act(observation: Dict[str, np.ndarray]) -> dict`:
      - Takes observations and returns actions for each agent
      - Input is dictionary: `{agent_name: observation}`
      - Output is dictionary: `{agent_name: action}`
   
   - `seed(seed)`:
     - Sets random seeds for reproducibility

3. Optionally override:
   - `comm(observation)`:
     - Generates messages sent to higher levels
     - Default implementation concatenates observations
     - Override to implement learned communication

Here's a minimal example showing the key components:

```python
from tame.hierarchy import LevelAgent
import gymnasium as gym
import numpy as np

class MyAgent(LevelAgent):
    def __init__(self, observation_space, action_space, communication_space, device, name="my_agent", args=None, torch_compile=False):
        self.observation_space = observation_space
        self.action_space = action_space
        self.communication_space = communication_space
        self.device = device
        self.name = name
        
        # Initialize your networks, buffers etc.
        self.memory = []
        
    def act_train(self, observation, global_step):
        # Add exploration during training
        if np.random.random() < 0.1:  # epsilon-greedy
            return {agent: self.action_space[agent].sample() for agent in observation}
        return self.act(observation)
        
    def act(self, observation):
        # Your action selection logic
        return {agent: self.action_space[agent].sample() for agent in observation}
        
    def store(self, state, action=None, reward=None, done=None):
        # Store transition in memory
        self.memory.append((state, action, reward, done))
        
    def update_step(self, global_step, writer):
        # Perform one training update
        if len(self.memory) >= batch_size:
            # Your training logic here
            if writer:
                writer.add_scalar("loss", loss, global_step)

When implementing your own agent, consider:
- Using existing implementations like `monolithic_ppo.py` as templates
- Testing agents independently before adding them to hierarchies
- Adding thorough logging during training for debugging
- Implementing proper cleanup in `__del__` if needed

The agent will be automatically integrated into training and evaluation flows by the `LevelEnv`, so focus on implementing the core learning algorithms and let the framework handle the hierarchical aspects.

### Note

1. A `LevelAgent` does not need the `train` function to be implemented, as its training is performed through the `update_step` function.
But if you implement `train` as well, you can use a `LevelAgent` also standalone with the `run_experiment.py` script.

2. In general, a `BaseAgent` is used to instantiate multiple `LevelAgent` through the hierarchy (example as in `src/tame/agents/ppo3.py`)

## Examples
You can find examples of how `BaseAgent` and `LevelAgent` are implemented in `src/tame/agents`.

## Next Steps

Now that you understand how to implement agents in TAG, you can:

1. Study example implementations in `src/tame/agents/`:
   - `monolithic_ppo.py` for a basic LevelAgent example
   - `ppo3.py` for a complete hierarchical agent

2. Check the full documentation for:
   - Advanced communication mechanisms between levels
   - Custom observation/action space configurations
   - Performance optimization tips
   - Debugging tools and techniques

3. Try building your own agent:
   - Start with a simple standalone BaseAgent
   - Convert it to a LevelAgent
   - Test it in increasingly complex hierarchies
   - Experiment with different communication strategies

4. Reference [Building your first hierarchy](tutorials/03_build_a_hierarchy.ipynb) to see how to integrate your agent into a complete hierarchical system.
